# Quick SLM — 03 · Multi-Checkpoint Capability Evaluation

Walks **every checkpoint** produced by `02_pretraining.ipynb`, and the SFT checkpoints from `05_sft_train.ipynb` when they exist, runs the same probe battery against each one, then uses **Gemma** as an LLM judge to score every output. Produces a table that shows how each capability emerges over training tokens, base and agent side by side.

This notebook does **not** touch `combined.bin` — perplexity is already tracked inside 02's training loop.

## Pipeline

1. Locate every checkpoint in `MyDrive/quick-slm/checkpoints/step_*` plus `final/`, and `sft_checkpoints/step_*` plus `sft_final/` if the SFT run has produced them. SFT is optional: absent SFT artifacts are simply not evaluated.
2. **Phase 1 — generate**: load each checkpoint → run all probes → save outputs to `logs/probe_outputs_<label>.json` → unload.
3. **Phase 2 — judge**: load Gemma once. Score every (prompt, expected, response) tuple 0-5 with a one-sentence reason. Cache to `logs/scored_outputs_<label>.json`.
4. **Phase 3 — table**: aggregate per checkpoint × category. Print a comparison table; save to `logs/eval_table.{json,csv}`.

## Probes

| Category | What it tests |
|---|---|
| Fluency | Locally coherent English from open-ended prompts |
| Knowledge | Factual recall — FineWeb-Edu coverage |
| Math | Arithmetic + comparison — FineMath coverage |
| Code | Python completion — StarCoder coverage |
| Tool | xLAM JSON shape + intent routing |

## Hardware

RTX PRO 6000 Blackwell (96 GB). Quick SLM (~5 GB BF16) and Gemma (~18 GB at 31B in 4-bit) load **sequentially** so memory stays predictable.

## Re-runnable

Both phases cache per-checkpoint files in `logs/`. Re-running adds new checkpoints without re-doing existing ones. Delete `probe_outputs_<label>.json` or `scored_outputs_<label>.json` to force a re-eval.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install dependencies

In [ ]:
# Don't touch torchvision/torchaudio — Colab ships them matched to torch's
# CUDA build. Just upgrade what we use plus pandas for the final table.
# bitsandbytes provides the 4-bit quantization for the Gemma judge.
!pip install -q --upgrade transformers accelerate safetensors tqdm pandas bitsandbytes

## 3. Locate all checkpoints + load tokenizer

Walks `checkpoints/step_*/hf` plus `final/`, then `sft_checkpoints/step_*/hf` plus `sft_final/` if the SFT run has produced them. Skips entries missing `config.json` (incomplete saves), and skips SFT entirely when its directories are absent. The tokenizer is shared across every checkpoint — base and SFT inherit the same one — so we load it once.

In [ ]:
import os, json, math, time
from pathlib import Path

DRIVE_ROOT    = Path('/content/drive/MyDrive/quick-slm')
CKPT_DIR      = DRIVE_ROOT / 'checkpoints'
FINAL_DIR     = DRIVE_ROOT / 'final'
SFT_CKPT_DIR  = DRIVE_ROOT / 'sft_checkpoints'   # written by 05_sft_train.ipynb
SFT_FINAL_DIR = DRIVE_ROOT / 'sft_final'         # the quick-slm-103m-agent artifact
LOGS_DIR      = DRIVE_ROOT / 'logs'
TOK_DIR       = DRIVE_ROOT / 'tokenizer'

# SFT steps sort after every pretraining step, and both finals sort after their
# own step series, so the table reads base -> base-final -> sft -> sft-final.
SFT_STEP_BASE  = 2 * 10**9
FINAL_STEP      = 10**9
SFT_FINAL_STEP  = 3 * 10**9

LOGS_DIR.mkdir(parents=True, exist_ok=True)

import torch
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32
print(f'device : {device}  dtype : {dtype}')
if device == 'cuda':
    print(f'gpu    : {torch.cuda.get_device_name(0)}')

# Tokenizer is shared across every Quick SLM checkpoint, base and SFT alike:
# SFT inherits the base tokenizer, so one load serves both.
tok = AutoTokenizer.from_pretrained(str(TOK_DIR))
print(f'tokenizer vocab : {len(tok)}')

def _walk_steps(ckpt_dir, label_prefix, step_offset):
    """Yield (label, step, hf_dir) for every complete step_* under ckpt_dir.

    Missing dir yields nothing, so an absent SFT run is simply not evaluated.
    """
    out = []
    if not ckpt_dir.exists():
        return out
    for c in sorted(ckpt_dir.glob('step_*'), key=lambda p: int(p.name.split('_')[1])):
        hf = (c / 'hf') if (c / 'hf').exists() else c
        if not (hf / 'config.json').exists():
            continue  # incomplete save
        step = int(c.name.split('_')[1])
        out.append((f'{label_prefix}step_{step:07d}', step_offset + step, hf))
    return out

def find_all_checkpoints():
    """Return list of (label, step, hf_dir) sorted by step.

    Base checkpoints first, then SFT checkpoints if 05_sft_train.ipynb has run.
    SFT is optional: if neither sft_checkpoints/ nor sft_final/ exists, only the
    base checkpoints are returned and nothing SFT-related is checked.
    """
    out = _walk_steps(CKPT_DIR, '', 0)
    if FINAL_DIR.exists() and (FINAL_DIR / 'config.json').exists():
        out.append(('final', FINAL_STEP, FINAL_DIR))

    # --- SFT, only if present ---
    out += _walk_steps(SFT_CKPT_DIR, 'sft_', SFT_STEP_BASE)
    if SFT_FINAL_DIR.exists() and (SFT_FINAL_DIR / 'config.json').exists():
        out.append(('sft_final', SFT_FINAL_STEP, SFT_FINAL_DIR))
    return out

CHECKPOINTS = find_all_checkpoints()
if not CHECKPOINTS:
    raise FileNotFoundError(f'No checkpoints found under {CKPT_DIR} or {FINAL_DIR}')
n_sft = sum(1 for label, _, _ in CHECKPOINTS if label.startswith('sft_'))
print(f'\nfound {len(CHECKPOINTS)} checkpoints ({len(CHECKPOINTS) - n_sft} base, {n_sft} SFT):')
for label, step, path in CHECKPOINTS:
    print(f'  {label:<20s}  step={step:>13,}  {path}')
if n_sft == 0:
    print('\n(no SFT artifacts on Drive yet — run 05_sft_train.ipynb to include the agent)')

## 4. Probe definitions

Each probe is `{category, prompt, expected, gen_args}`. `expected` is a free-form description (not a string-match) — it's passed to Gemma so the judge knows what a good answer looks like. Tool-calling prompts get the `<tool>...</tool>` block injected so the model sees the same template it was trained on.

In [4]:
import json as _json

# Tool-calling helper — built once, reused below
_TOOLS_DEMO = [
    {'name': 'get_weather', 'description': 'Get current weather for a city',
     'parameters': {'type': 'object',
                    'properties': {'city': {'type': 'string'}},
                    'required': ['city']}},
    {'name': 'add_numbers', 'description': 'Add two integers',
     'parameters': {'type': 'object',
                    'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
                    'required': ['a', 'b']}},
]

def _tool_prompt(query):
    return (f'User request: {query}\n'
            f'<tool> Available tools: {_json.dumps(_TOOLS_DEMO)} </tool>\n'
            f'<response> Valid function calls: ')

PROBES = [
    # --- Fluency: locally coherent English ---
    {'category': 'fluency', 'prompt': 'The quick brown fox',
     'expected': 'A continuation that maintains the subject (fox) and produces coherent English.',
     'gen_args': {'max_new': 60, 'temperature': 0.4}},
    {'category': 'fluency', 'prompt': 'In a small village near the mountains,',
     'expected': 'A coherent narrative about a village or surrounding mountains.',
     'gen_args': {'max_new': 60, 'temperature': 0.4}},
    {'category': 'fluency', 'prompt': 'Photosynthesis is the process by which plants',
     'expected': 'Plants convert light/CO2/water into sugars/energy/oxygen.',
     'gen_args': {'max_new': 60, 'temperature': 0.4}},
    {'category': 'fluency', 'prompt': 'When you compare two numbers, the larger one',
     'expected': 'A coherent ordering statement, e.g., "is greater than the smaller one".',
     'gen_args': {'max_new': 60, 'temperature': 0.4}},

    # --- Knowledge: factual recall ---
    {'category': 'knowledge', 'prompt': 'The capital of France is',
     'expected': 'Paris.',
     'gen_args': {'max_new': 15, 'temperature': 0.0}},
    {'category': 'knowledge', 'prompt': 'The boiling point of water at sea level is',
     'expected': '100 degrees Celsius (or 212 Fahrenheit).',
     'gen_args': {'max_new': 15, 'temperature': 0.0}},
    {'category': 'knowledge', 'prompt': 'Albert Einstein is best known for his theory of',
     'expected': 'Relativity (special or general).',
     'gen_args': {'max_new': 15, 'temperature': 0.0}},
    {'category': 'knowledge', 'prompt': 'The chemical symbol for gold is',
     'expected': 'Au.',
     'gen_args': {'max_new': 15, 'temperature': 0.0}},
    {'category': 'knowledge', 'prompt': 'The largest planet in our solar system is',
     'expected': 'Jupiter.',
     'gen_args': {'max_new': 15, 'temperature': 0.0}},
    {'category': 'knowledge', 'prompt': 'William Shakespeare wrote the play',
     'expected': 'A Shakespeare play title (Hamlet, Macbeth, Romeo and Juliet, etc.).',
     'gen_args': {'max_new': 15, 'temperature': 0.0}},

    # --- Math: numeric reasoning ---
    {'category': 'math', 'prompt': 'Question: Which is larger, 23 or 25?\nAnswer: ',
     'expected': '25.',
     'gen_args': {'max_new': 24, 'temperature': 0.0}},
    {'category': 'math', 'prompt': 'Question: 5 + 3 = ?\nAnswer: ',
     'expected': '8.',
     'gen_args': {'max_new': 24, 'temperature': 0.0}},
    {'category': 'math', 'prompt': 'Question: 12 - 7 = ?\nAnswer: ',
     'expected': '5.',
     'gen_args': {'max_new': 24, 'temperature': 0.0}},
    {'category': 'math', 'prompt': 'Question: 4 × 6 = ?\nAnswer: ',
     'expected': '24.',
     'gen_args': {'max_new': 24, 'temperature': 0.0}},
    {'category': 'math', 'prompt': 'Question: Half of 18 is\nAnswer: ',
     'expected': '9.',
     'gen_args': {'max_new': 24, 'temperature': 0.0}},

    # --- Code: Python completion ---
    {'category': 'code', 'prompt': 'def fibonacci(n):\n    if n <= 1:\n        return n\n    return ',
     'expected': 'fibonacci(n-1) + fibonacci(n-2) — the recursive Fibonacci formula.',
     'gen_args': {'max_new': 40, 'temperature': 0.0}},
    {'category': 'code', 'prompt': 'def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, ',
     'expected': 'A loop bound such as int(n**0.5)+1 with a divisibility check returning False inside.',
     'gen_args': {'max_new': 40, 'temperature': 0.0}},
    {'category': 'code', 'prompt': 'import numpy as np\n\n# Compute the mean of an array\ndef mean(arr):\n    return ',
     'expected': 'sum(arr)/len(arr), np.mean(arr), or arr.mean() — any correct mean expression.',
     'gen_args': {'max_new': 40, 'temperature': 0.0}},

    # --- Tool-calling: xLAM JSON shape + intent routing ---
    {'category': 'tool',
     'prompt': _tool_prompt('What is the weather in Paris?'),
     'user_query': 'What is the weather in Paris?',
     'expected': '[{"name": "get_weather", "arguments": {"city": "Paris"}}] — correct tool name AND city argument bound to "Paris".',
     'gen_args': {'max_new': 80, 'temperature': 0.0, 'stop': ['</response>']}},
    {'category': 'tool',
     'prompt': _tool_prompt('Please add 17 and 25.'),
     'user_query': 'Please add 17 and 25.',
     'expected': '[{"name": "add_numbers", "arguments": {"a": 17, "b": 25}}] — correct tool name AND both numeric arguments bound.',
     'gen_args': {'max_new': 80, 'temperature': 0.0, 'stop': ['</response>']}},
]

from collections import Counter
counts = Counter(p['category'] for p in PROBES)
print(f'{len(PROBES)} probes:')
for cat, n in sorted(counts.items()):
    print(f'  {cat:<10s} {n}')

20 probes:
  code       3
  fluency    4
  knowledge  6
  math       5
  tool       2


## 5. Generation utility

`generate(model, prompt, **gen_args)` — runs Quick SLM or any HF causal LM. Adds `repetition_penalty=1.1` to cut the worst attractor loops on early checkpoints (doesn't affect later ones since the LM picks more diverse tokens once trained).

In [5]:
@torch.no_grad()
def generate(model, prompt, *, max_new=80, temperature=0.0, top_p=0.9, stop=None):
    """Generate from any HF causal LM. Returns only the new tokens, decoded.

    repetition_penalty=1.1 cuts attractor loops on under-trained checkpoints
    without meaningfully changing well-trained ones.
    """
    enc = tok(prompt, return_tensors='pt').to(model.device)
    prompt_len = enc.input_ids.shape[1]
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=top_p,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
        repetition_penalty=1.1,
    )
    text = tok.decode(out[0, prompt_len:], skip_special_tokens=False)
    if stop:
        for s in stop:
            i = text.find(s)
            if i != -1:
                text = text[:i]
    return text.strip()

## 6. Phase 1 — Generate from every checkpoint

For each checkpoint: load → run all 20 probes → save outputs to `logs/probe_outputs_<label>.json` → unload. **Cached** — already-evaluated checkpoints are skipped on re-runs.

Memory pattern: load Quick SLM (~5 GB BF16) → ~20 short generations → unload. ~1 minute per checkpoint on RTX PRO 6000.

In [6]:
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm

def probe_outputs_path(label):
    return LOGS_DIR / f'probe_outputs_{label}.json'

def run_probes_on_checkpoint(label, hf_dir):
    """Load checkpoint, run all probes, return dict {label, hf_dir, outputs}."""
    out_path = probe_outputs_path(label)
    if out_path.exists():
        return json.loads(out_path.read_text())

    print(f'\n  loading {hf_dir}')
    try:
        model = AutoModelForCausalLM.from_pretrained(
            str(hf_dir), torch_dtype=dtype, attn_implementation='sdpa'
        ).to(device)
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(str(hf_dir), torch_dtype=dtype).to(device)
    model.eval()

    outputs = []
    for p in tqdm(PROBES, desc=f'  {label}', leave=False):
        gen_args = p.get('gen_args', {})
        try:
            response = generate(model, p['prompt'], **gen_args)
        except Exception as e:
            response = f'[ERROR: {type(e).__name__}: {e}]'
        outputs.append({
            'category': p['category'],
            'prompt': p['prompt'],
            'expected': p['expected'],
            'response': response,
            'user_query': p.get('user_query'),
        })

    record = {'label': label, 'hf_dir': str(hf_dir), 'outputs': outputs}
    out_path.write_text(json.dumps(record, indent=2))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record

ALL_OUTPUTS = []
for label, step, path in CHECKPOINTS:
    rec = run_probes_on_checkpoint(label, path)
    ALL_OUTPUTS.append(rec)

print(f'\ngenerated outputs for {len(ALL_OUTPUTS)} checkpoints')
print(f'cache dir: {LOGS_DIR}/probe_outputs_*.json')


  loading /content/drive/MyDrive/quick-slm/checkpoints/step_0006000/hf


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:01<?, ?it/s]

  step_0006000:   0%|          | 0/20 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



  loading /content/drive/MyDrive/quick-slm/checkpoints/step_0006500/hf


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

  step_0006500:   0%|          | 0/20 [00:00<?, ?it/s]


generated outputs for 13 checkpoints
cache dir: /content/drive/MyDrive/quick-slm/logs/probe_outputs_*.json


## 7. Phase 2 — Load the Gemma judge

Loads Google's QAT Q4 checkpoint **`google/gemma-4-31B-it-qat-q4_0-unquantized`** as the LLM judge. Each `(prompt, expected, response)` tuple is scored 0-5 with a one-sentence reason.

This is a **dense** 31B model — all 31B parameters are active per token. Loaded in **4-bit** nf4 via bitsandbytes: ~18 GB in VRAM versus ~62 GB at BF16. BF16 is banned for this model — its weights alone leave too little headroom on the RTX PRO 6000 (96 GB) once activations and the KV cache are added. Quantization-aware training means the Q4 weights were tuned for 4-bit, so the impact on judge quality is small. Compute throughput is lower than a small MoE, so judging is slower per call, but a stronger judge gives more reliable scores.

Open access — **no HuggingFace token needed**. The `-unquantized` repo ships QAT-tuned weights that `transformers` loads and bitsandbytes re-quantizes to 4-bit on the way in, so the first run still downloads the full weights once (a few minutes on a fast connection).

In [ ]:
import re
from transformers import BitsAndBytesConfig

# Open-access dense judge: Gemma 4 31B, from Google's QAT Q4 checkpoint, loaded
# in 4-bit nf4 (~18 GB in VRAM). bf16 is banned for this model: the full weights
# are ~62 GB, which OOMs the card once activations + the KV cache are added.
# Quantization-aware training means the Q4 weights were tuned for 4-bit, so the
# impact on judge quality is small. All 31B params active per token — slower than
# a small MoE, but a stronger judge gives more reliable scores.
GEMMA_MODEL = 'google/gemma-4-31B-it-qat-q4_0-unquantized'

# 4-bit nf4 via bitsandbytes; device_map='auto' places the quantized weights on
# the GPU. Compute stays bf16. Don't pass a plain dtype alongside this.
_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'loading judge: {GEMMA_MODEL} (4-bit nf4)')
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_MODEL)
gemma     = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL, quantization_config=_quant_cfg, device_map='auto'
)
gemma.eval()
total_b  = sum(p.numel() for p in gemma.parameters()) / 1e9
print(f'  loaded — {total_b:.1f} B params (dense)')

JUDGE_RUBRIC = """You are evaluating outputs from a small (103M parameter) language model that's still in pretraining. The model has not been fine-tuned, so most outputs will be poor.

Score the response from 0 to 5:
- 0: nonsensical, empty, or pure repetition loop
- 1: grammatical English but topically unrelated to the prompt
- 2: on-topic but factually or structurally wrong
- 3: partially correct or relevant; missing or wrong in specific places
- 4: mostly correct with minor flaws
- 5: fully correct and well-formed

Be strict. Most outputs from an early-training model should score 0-2.

PROMPT:
{prompt}

EXPECTED (what a good answer looks like):
{expected}

MODEL RESPONSE:
{response}

Respond with EXACTLY this format and nothing else:
SCORE: <integer 0-5>
REASON: <one short sentence>"""

_SCORE_RE  = re.compile(r'SCORE:\s*(\d+)', re.IGNORECASE)
_REASON_RE = re.compile(r'REASON:\s*(.+)', re.IGNORECASE)

@torch.no_grad()
def gemma_score(prompt, expected, response):
    """Run Gemma judge on one (prompt, expected, response). Returns (score:int, reason:str).

    score = -1 if Gemma's output didn't parse.
    Truncates response to 800 chars to keep judge prompts bounded.
    """
    msg = JUDGE_RUBRIC.format(prompt=prompt[:800], expected=expected, response=response[:800])
    chat = [{'role': 'user', 'content': msg}]

    # Two-step: chat template → text → tokenize. Robust across transformers
    # versions (newer ones return BatchEncoding from apply_chat_template,
    # older ones return raw tensors — splitting avoids the .shape pitfall).
    text = gemma_tok.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    enc = gemma_tok(text, return_tensors='pt').to(gemma.device)
    prompt_len = enc['input_ids'].shape[1]

    out = gemma.generate(
        **enc,
        max_new_tokens=120, do_sample=False,
        pad_token_id=gemma_tok.pad_token_id or gemma_tok.eos_token_id,
    )
    decoded = gemma_tok.decode(out[0, prompt_len:], skip_special_tokens=True)
    score_m  = _SCORE_RE.search(decoded)
    reason_m = _REASON_RE.search(decoded)
    score    = int(score_m.group(1)) if score_m else -1
    reason   = reason_m.group(1).strip() if reason_m else decoded.strip()[:200]
    return max(-1, min(score, 5)), reason

# Quick smoke test on a known case
_s, _r = gemma_score('The capital of France is', 'Paris.', 'Paris.')
print(f'\nsmoke test  : score={_s}  reason={_r!r}')

## 8. Run grading on every (checkpoint × probe) tuple

20 probes × N checkpoints. Each grading call is a few seconds on the 31B judge. Cached per checkpoint to `logs/scored_outputs_<label>.json` — re-runs only score new checkpoints.

In [ ]:
def scored_path(label):
    return LOGS_DIR / f'scored_outputs_{label}.json'

ALL_SCORED = []
for record in ALL_OUTPUTS:
    label = record['label']
    out_path = scored_path(label)
    if out_path.exists():
        ALL_SCORED.append(json.loads(out_path.read_text()))
        continue

    print(f'\nscoring {label} ({len(record["outputs"])} probes)')
    scored = {'label': label, 'hf_dir': record['hf_dir'], 'judge': GEMMA_MODEL, 'outputs': []}
    for o in tqdm(record['outputs'], desc=f'  {label}', leave=False):
        score, reason = gemma_score(o['prompt'], o['expected'], o['response'])
        scored['outputs'].append({**o, 'score': score, 'reason': reason})
    out_path.write_text(json.dumps(scored, indent=2))
    ALL_SCORED.append(scored)

print(f'\nscored {len(ALL_SCORED)} checkpoints')
print(f'cache dir: {LOGS_DIR}/scored_outputs_*.json')

## 9. Phase 3 — Comparison table

Aggregates Gemma scores per checkpoint × category. Each cell is the **average** of the probes in that category for that checkpoint (0-5 scale). The `overall` column averages across all categories. Saved to `logs/eval_table.{json,csv}` for plotting later.

In [ ]:
import pandas as pd
from collections import defaultdict

CATEGORIES = ['fluency', 'knowledge', 'math', 'code', 'tool']

def step_of(label):
    # Mirror the step values find_all_checkpoints() assigned, so the table
    # orders base -> base-final -> sft -> sft-final off the label alone.
    if label == 'final':
        return 10**9
    if label == 'sft_final':
        return 3 * 10**9
    if label.startswith('sft_step_'):
        return 2 * 10**9 + int(label.split('_')[-1])
    return int(label.split('_')[-1])

rows = []
for s in ALL_SCORED:
    by_cat = defaultdict(list)
    for o in s['outputs']:
        if o.get('score', -1) >= 0:
            by_cat[o['category']].append(o['score'])
    row = {'label': s['label'], 'step': step_of(s['label'])}
    all_scores = []
    for cat in CATEGORIES:
        if by_cat[cat]:
            row[cat] = round(sum(by_cat[cat]) / len(by_cat[cat]), 2)
            all_scores.extend(by_cat[cat])
        else:
            row[cat] = None
    row['overall'] = round(sum(all_scores) / max(len(all_scores), 1), 2) if all_scores else None
    rows.append(row)

df = pd.DataFrame(rows).sort_values('step').reset_index(drop=True)
display_cols = ['label', 'step'] + CATEGORIES + ['overall']
print('\nQuick SLM — capability emergence across checkpoints (Gemma 0-5 scores)')
print('=' * 96)
print(df[display_cols].to_string(index=False))
print('=' * 96)

(LOGS_DIR / 'eval_table.json').write_text(df.to_json(orient='records', indent=2))
df.to_csv(LOGS_DIR / 'eval_table.csv', index=False)
print(f'\nsaved: {LOGS_DIR}/eval_table.json')
print(f'saved: {LOGS_DIR}/eval_table.csv')